In [2]:
from __future__ import annotations

import os
# os.environ["HWLOC_HIDE_ERRORS"] = "1"
# os.environ["TF_ENABLE_NUMA"] = "0"
# os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".80"
# 3. Provide valid temporary directory for ptxas CUDA kernel compilation
os.environ["TMPDIR"] = "/tmp"
os.environ["TMP"] = "/tmp"
os.environ["TEMP"] = "/tmp"

import numpy as np
import jax
jax.config.update("jax_enable_x64", True)
print("Jax version", jax.__version__, "64-bit", jax.config.read("jax_enable_x64"))
print("jax", jax.__version__, "GPU backend:", jax.default_backend() == 'gpu', f"({jax.devices('gpu')[0].device_kind})" if jax.default_backend() == 'gpu' else "")
import jax.numpy as jnp
import s2fft
from tqdm.auto import tqdm
from typing import Literal

import jax.numpy as jnp
import s2fft
import argparse
import shutil
from pathlib import Path

Jax version 0.11.0 64-bit True
jax 0.11.0 GPU backend: True (NVIDIA GH200 120GB)


In [3]:
SOURCE = "/projects/u6pf/brianycc/spherical_maps/"
OUTPUT = "/projects/u6pf/brianycc/spherical_maps/5_tomo_bins_flm_750_all_mwss/"
SEARCH_BATCH = "5_tomo_bins_flm_750_batch_*_mwss/"
SEARCH_MAP = "map_*.npy"

params = []
Path(OUTPUT).mkdir(parents=True, exist_ok=True)
map_count = 0

BATCH_DIRS = sorted(Path(SOURCE).glob(SEARCH_BATCH),
    key=lambda p: int(p.name.split("_")[-2]) # Sort by batch number, which is the 3rd last part of the directory name
    )
for batch_dir in BATCH_DIRS:

    map_files = sorted(batch_dir.glob(SEARCH_MAP), 
        key=lambda p: int(p.stem.split("_")[-1]) # Sort by map index, which is the last part of the filename after the last underscore
        )
    print(f"Batch {batch_dir.name}: {len(map_files)} maps found.")

    original_params = np.load(batch_dir / "params.npy")
    
    for map_file in map_files:

        original_index = int(map_file.stem.split("_")[-1])
        new_index = map_count

        map_count += 1

        params.append(original_params[original_index])

        output_path = Path(OUTPUT) / f"map_{new_index}.npy"
        if output_path.exists():
            print(f"Skipping existing file: {output_path.name}")
            continue
 
        shutil.copy(map_file, output_path)

params_array = np.array(params)
np.save(Path(OUTPUT) / "params.npy", params_array)
print(f"Processed {len(params_array)} parameters and {map_count} map files.")

Batch 5_tomo_bins_flm_750_batch_1_mwss: 2000 maps found.
Batch 5_tomo_bins_flm_750_batch_2_mwss: 2000 maps found.
Batch 5_tomo_bins_flm_750_batch_3_mwss: 2000 maps found.
Processed 6000 parameters and 6000 map files.


In [ ]:
BATCH_DIR.glob("map_*.npy")